# PR10: Dimensionality Reduction using Principal Component Analysis (PCA) (CO4)

**Aim:** Apply PCA to a high-dimensional dataset and compare the number of features before and after PCA, visualize reduced dimensions, and evaluate the impact on classification performance.

**Dataset:** Breast Cancer Wisconsin dataset from Scikit-learn.

## 1. Objectives

1. Understand dimensionality reduction using PCA.
2. Reduce the number of features in a high-dimensional dataset.
3. Visualize the dataset in two-dimensional space using PCA.
4. Compare classification performance before and after PCA.

## 2. Theory

Principal Component Analysis (PCA) is an unsupervised dimensionality reduction technique that transforms correlated features into a smaller set of uncorrelated variables called **principal components**. PCA selects components that preserve as much data variance as possible.

**Main steps:**
- Standardize the features.
- Fit PCA on the training data.
- Select components based on explained variance.
- Transform the data into the reduced feature space.
- Compare classification performance before and after PCA.

## 3. Import Libraries

Run this cell first to import the required libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

## 4. Load the Dataset

In [ ]:
# Load the Breast Cancer Wisconsin dataset
data = load_breast_cancer()
X = data.data
y = data.target

print("Dataset shape:", X.shape)
print("Number of samples:", X.shape[0])
print("Original number of features:", X.shape[1])
print("Target classes:", data.target_names)

## 5. Split the Dataset into Training and Testing Sets

In [ ]:
# Use 80% of the data for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 6. Classification Before PCA

Standardize the data and train Logistic Regression using all original features. The scaler is fitted only on the training data.

In [ ]:
model_original = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000))
])

model_original.fit(X_train, y_train)
y_pred_original = model_original.predict(X_test)

accuracy_original = accuracy_score(y_test, y_pred_original)

print(f"Original number of features: {X_train.shape[1]}")
print(f"Accuracy without PCA: {accuracy_original * 100:.2f}%")

## 7. Apply PCA and Classify

Retain 95% of the variance. PCA and scaling are fitted only on the training data, preventing test-data leakage.

In [ ]:
pca_model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("classifier", LogisticRegression(max_iter=5000))
])

pca_model.fit(X_train, y_train)
y_pred_pca = pca_model.predict(X_test)

accuracy_pca = accuracy_score(y_test, y_pred_pca)
pca = pca_model.named_steps["pca"]

print("Original number of features:", X_train.shape[1])
print("Number of features after PCA:", pca.n_components_)
print(f"Explained variance retained: {pca.explained_variance_ratio_.sum() * 100:.2f}%")
print(f"Accuracy with PCA: {accuracy_pca * 100:.2f}%")

## 8. Compare Classification Performance

In [ ]:
print("Classification Performance Comparison")
print("--------------------------------------")
print(f"Accuracy without PCA: {accuracy_original * 100:.2f}%")
print(f"Accuracy with PCA:    {accuracy_pca * 100:.2f}%")
print(f"Accuracy difference:  {(accuracy_pca - accuracy_original) * 100:+.2f} percentage points")

print("\nClassification report after PCA:")
print(classification_report(
    y_test,
    y_pred_pca,
    target_names=data.target_names
))

## 9. Visualize the Dataset in Two Dimensions

For visualization only, fit a scaler and a two-component PCA transformation to the complete dataset. This plot is exploratory; it is not used to calculate classification accuracy.

In [ ]:
X_scaled_visual = StandardScaler().fit_transform(X)

pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled_visual)

plt.figure(figsize=(8, 6))

for class_id, class_name in enumerate(data.target_names):
    plt.scatter(
        X_2d[y == class_id, 0],
        X_2d[y == class_id, 1],
        label=class_name,
        alpha=0.7
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA: Breast Cancer Dataset in 2D")
plt.legend()
plt.grid(True)
plt.show()

print(f"Variance explained by first two components: {pca_2d.explained_variance_ratio_.sum() * 100:.2f}%")

## 10. Plot Cumulative Explained Variance

In [ ]:
# Fit PCA on the standardized training set to inspect variance retention
X_train_scaled = StandardScaler().fit_transform(X_train)
pca_full = PCA().fit(X_train_scaled)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o"
)
plt.axhline(y=0.95, linestyle="--", label="95% variance threshold")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.legend()
plt.grid(True)
plt.show()

components_for_95 = np.argmax(cumulative_variance >= 0.95) + 1
print("Components needed to retain at least 95% variance:", components_for_95)

## 11. Summary Table

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Measure": [
        "Number of features",
        "Test accuracy"
    ],
    "Before PCA": [
        X_train.shape[1],
        f"{accuracy_original * 100:.2f}%"
    ],
    "After PCA": [
        pca.n_components_,
        f"{accuracy_pca * 100:.2f}%"
    ]
})

comparison

## 12. Result

PCA was applied to the Breast Cancer Wisconsin dataset. The original feature count, reduced feature count, explained variance, and Logistic Regression accuracy before and after PCA were compared. The actual numerical results are displayed by the notebook cells when executed.

## 13. Conclusion

PCA reduces the number of dimensions while preserving a selected proportion of variance. The two-dimensional visualization helps inspect class separation, and the classification comparison shows how dimensionality reduction affects predictive performance.